In [ ]:
# SOme of these file paths are wonky in this and the other API file in this folder, 
# I'm not sure if I changed them all yet so please do not actually run any extraction. 
import requests
import json
import os
import urllib3
import boto3
import io
import gzip
import logging
import pandas as pd
from io import StringIO
from datetime import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar

In [ ]:
def fetch_tomtom_traffic_data():
    """
    Fetches traffic data from the TomTom API for specified latitude and longitude coordinates and returns a DataFrame containing the relevant information.

    Args: None, the latitude and longitude coordinates are hardcoded in the function.

    Returns: tomtom_df (DataFrame): A DataFrame containing the traffic data for the specified coordinates, including timestamp, latitude, longitude, current speed, free flow speed, current delay, and confidence.
    """
     
    api_key = 'Ask_Bella'


    url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"

    data_list = []

    latitude = [35.307778, 35.313611, 35.311361, 35.309306, 35.306917, 35.293111, 35.294889, 35.299639, 35.287972, 35.301861, 35.311861] # fill in with the found values
    longitude = [-80.724583, -80.701944, -80.711222, -80.718056, -80.722639, -80.749361, -80.742111, -80.74575, -80.759472, -80.752333, -80.744]
    # Loop through the list of coordinates and fetch traffic data for each location
    for i in range(len(latitude)):

            params = {
                'key': api_key,
                'point': f"{latitude[i]},{longitude[i]}",
                'unit': "MPH",
                "openLr": "false"
            }

            response = requests.get(url, params)

            if response.status_code==200:
                data = response.json()
                flow_data = data.get("flowSegmentData", {})

                data_we_want = {
                    "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
                    "latitude": latitude[i],
                    "longitude": longitude[i],
                    "current_speed": flow_data.get("currentSpeed"),
                    "free_flow_speed": flow_data.get("freeFlowSpeed"),
                    "current_delay": flow_data.get("currentDelay"),
                    "confidence": flow_data.get("confidence") # for flags of when we should somehow retrain?
                }

                data_list.append(data_we_want)
            else:
                print(f"Error in retrieving data: {response.status_code}")
                
    #making data_list into a DataFrame
    tomtom_df = pd.DataFrame(data_list) 
    return tomtom_df

             timestamp   latitude  longitude  current_speed  free_flow_speed  \
0  2026-09-27_15-05-39  35.307778 -80.724583             27               27   
1  2026-09-27_15-05-39  35.313611 -80.701944             40               40   
2  2026-09-27_15-05-39  35.311361 -80.711222             25               31   
3  2026-09-27_15-05-39  35.309306 -80.718056             25               31   
4  2026-09-27_15-05-39  35.306917 -80.722639             25               31   

  current_delay  confidence  
0          None           1  
1          None           1  
2          None           1  
3          None           1  
4          None           1  


In [ ]:
def edit_time_features(tomtom_df):
    """
    Adds time features to the TomTom traffic DataFrame; including hour, day of the week, month, and speed ratio as properties of existing features (timestamp and current_speed vs. free_flow_speed).

    Args:
        tomtom_df (DataFrame): A DataFrame containing the traffic data.
    
    Returns:
        tomtom_df (DataFrame): The DataFrame with the added features. speed_ratio is calculated as current_speed divided by free_flow_speed. The time features are extracted from the timestamp include hour, day_of_week, and month.
    """
    
    tomtom_df["timestamp"] = pd.to_datetime(tomtom_df["timestamp"])
    # Create time features
    tomtom_df["hour"] = tomtom_df["timestamp"].dt.hour
    tomtom_df["day_of_week"] = tomtom_df["timestamp"].dt.dayofweek
    tomtom_df["month"] = tomtom_df["timestamp"].dt.month

    # Calculate the speed ratio for each traffic record
    tomtom_df["speed_ratio"] = (tomtom_df["current_speed"] / tomtom_df["free_flow_speed"])

    return tomtom_df

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month
0,2026-09-27 15:05:39,35.307778,-80.724583,27,27,None,1,6,15,9
1,2026-09-27 15:05:39,35.313611,-80.701944,40,40,None,1,6,15,9
2,2026-09-27 15:05:39,35.311361,-80.711222,25,31,None,1,6,15,9
3,2026-09-27 15:05:39,35.309306,-80.718056,25,31,None,1,6,15,9
4,2026-09-27 15:05:39,35.306917,-80.722639,25,31,None,1,6,15,9
5,2026-09-27 15:05:40,35.293111,-80.749361,18,29,None,1,6,15,9
6,2026-09-27 15:05:40,35.294889,-80.742111,55,55,None,1,6,15,9
7,2026-09-27 15:05:40,35.299639,-80.745750,28,28,None,1,6,15,9
8,2026-09-27 15:05:40,35.287972,-80.759472,25,25,None,1,6,15,9
9,2026-09-27 15:05:40,35.301861,-80.752333,27,27,None,1,6,15,9


In [ ]:
def add_holiday_info(tomtom_df):
    """
    Adds holiday information to the TomTom traffic DataFrame by mapping timestamps to US federal holidays.

    Args:
        tomtom_df (DataFrame): A DataFrame containing the traffic data.
    
    Returns:
        tomtom_df (DataFrame): The DataFrame with an added 'holiday' column indicating whether each timestamp falls on a US federal holiday or not.
    """

    calendar = USFederalHolidayCalendar()
        #To use after getting traffic data from holidays
    holiday_names = calendar.holidays(
            start=tomtom_df["timestamp"].min().normalize(),
            end=tomtom_df["timestamp"].max().normalize(),
            return_name=True,
        )

    tomtom_df["holiday"] = (
            #map timestamp to the correct holiday name
            tomtom_df["timestamp"].dt.normalize()
            .map(holiday_names)
            .fillna("Not a federal holiday")
        )
    if not tomtom_df.empty:
        holiday_names = calendar.holidays(
            start=tomtom_df["timestamp"].min().normalize(),
            end=tomtom_df["timestamp"].max().normalize(),
            return_name=True,
        )

        tomtom_df["holiday"] = (
            tomtom_df["timestamp"].dt.normalize()
            .map(holiday_names)
            .fillna("Not a federal holiday")
        )
    else:
        print("No traffic rows available.")

    return tomtom_df

In [ ]:
def save_tomtom_data_to_s3(tomtom_df):
    """
    Saves the traffic information DataFrame to an S3 bucket in CSV format with the current timestamp.

    Args:
        tomtom_df: A DataFrame containing traffic data (NCDOT average annual traffic counts).
    
    Returns:
        None
    """
    # Include date and time
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S-%f")
    file_key = f"tomtom/traffic/{timestamp}.csv"

    csv_buffer = StringIO()
    tomtom_df.to_csv(csv_buffer, index=False)
    
    csv_string = csv_buffer.getvalue()
    bucket = "s3-3601-final-project"

    # Upload to S3
    s3_client.put_object(
        Bucket=bucket,
        Key=file_key,
        Body=csv_string.encode("utf-8"),
        ContentType="text/csv",
    )

    print("Saved file:")
    print(f"s3://{bucket}/{file_key}")

{'ResponseMetadata': {'RequestId': 'R08XAEEP1VB92Y1R',
  'HostId': 'Kfn9q5H2glrI+hbvvTzP8Ipy4NfaKCnIwFe4bhTd5TbXVORniICSf8Mct+N04WXdqKVmb9FUNis=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'Kfn9q5H2glrI+hbvvTzP8Ipy4NfaKCnIwFe4bhTd5TbXVORniICSf8Mct+N04WXdqKVmb9FUNis=',
   'x-amz-request-id': 'R08XAEEP1VB92Y1R',
   'date': 'Sun, 27 Sep 2026 15:06:02 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"da5ba3adb342c9750b10da4020cd961e"',
   'x-amz-checksum-crc32': 'hB/PPA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"da5ba3adb342c9750b10da4020cd961e"',
 'ChecksumCRC32': 'hB/PPA==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [ ]:
s3_client = boto3.client("s3")

tomtom_df = fetch_tomtom_traffic_data()
edit_time_features(tomtom_df)
add_holiday_info(tomtom_df)
save_tomtom_data_to_s3(tomtom_df)